# 064 — GRU

The GRU asks whether the LSTM needs everything it has: it merges the forget and
input gates into one, and drops the separate cell state. Two gates, one state,
**exactly 25% fewer parameters** — and on lesson 060's experiment **19 steps of
memory against the LSTM's 15**. The same catch applies: at update bias 0 it
manages **8 steps, below the plain RNN's 9**.

| Part | What we check |
|---|---|
| A | one GRU step, three lines |
| B | gradient flow against the update-gate bias |
| C | all three architectures side by side |
| D | parameters, and what the saving buys in wall clock |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — One GRU step

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

F, H = 8, 20
rng = np.random.default_rng(0)
W = {g: rng.normal(0, 0.3, (F + H, H)) for g in "rzc"}
b = {g: np.zeros(H) for g in "rzc"}

x, h = rng.normal(0, 1, F), np.zeros(H)
z_in = np.concatenate([h, x])
reset = sigmoid(z_in @ W["r"] + b["r"])                      # 1. reset gate
update = sigmoid(z_in @ W["z"] + b["z"])                     # 2. update gate
cand = np.tanh(np.concatenate([reset * h, x]) @ W["c"] + b["c"])   # 3. candidate
h_new = (1 - update) * h + update * cand                     #    one state, no C

print(f"  reset  mean {reset.mean():+.3f}")
print(f"  update mean {update.mean():+.3f}   carry (1-z) mean "
      f"{(1 - update).mean():+.3f}")
print(f"  new state shape {h_new.shape}  -  there is no cell state")
assert h_new.shape == (H,)
print()
print("One gate decides how much to keep AND how much to write: (1-z) and z")
print("sum to 1, which is the merge. Clearing a slot and writing to it are")
print("treated as the same decision.")

## Part B — Gradient flow

In [ ]:
def gru_flow(steps, update_bias, seed=0):
    r = np.random.default_rng(seed)
    Wl = {g: r.normal(0, 0.3, (F + H, H)) for g in "rzc"}
    bl = {g: np.zeros(H) for g in "rzc"}
    bl["z"] = bl["z"] + update_bias            # negative -> small z -> large carry
    X = r.normal(0, 1, (steps, F))
    hh, carries = np.zeros(H), []
    for t in range(steps):
        zz = np.concatenate([hh, X[t]])
        rr = sigmoid(zz @ Wl["r"] + bl["r"])
        uu = sigmoid(zz @ Wl["z"] + bl["z"])
        cc = np.tanh(np.concatenate([rr * hh, X[t]]) @ Wl["c"] + bl["c"])
        hh = (1 - uu) * hh + uu * cc
        carries.append(1 - uu)
    dh, norms = np.ones(H), []
    for t in range(steps - 1, -1, -1):
        norms.append(np.linalg.norm(dh))
        dh = dh * carries[t]                   # the entire backward path
    return np.array(norms[::-1]), float(np.mean([c.mean() for c in carries]))

def memory(norms):
    relative = norms[::-1] / norms[-1]
    below = np.where(relative < 0.01)[0]
    return int(below[0]) if len(below) else None

print(f"  {'update bias':>12}{'mean (1-z)':>12}{'99 back':>12}{'memory':>8}")
mem = {}
for bias in (0.0, -1.0, -2.0, -3.0):
    norms, carry = gru_flow(100, bias)
    mem[bias] = memory(norms)
    print(f"  {bias:>12.1f}{carry:>12.3f}{norms[0]:>12.2e}{mem[bias]:>8}")
assert mem[0.0] == 8 and mem[-1.0] == 19
print("\nSame structure as the LSTM: an exponential whose base the network picks.")

## Part C — All three, honestly

The RNN row is lesson 060's $\rho = 0.9$ run: gradient 5.1e-26, **9 steps** at
the 1% threshold. The LSTM row is lesson 062's forget bias 1.

In [ ]:
RNN_GRADIENT, RNN_MEMORY = 5.13e-26, 9
LSTM_GRADIENT, LSTM_MEMORY = 1.71e-13, 15

gru_norms, _ = gru_flow(100, -1.0)
rows = [("simple RNN (rho = 0.9)", RNN_GRADIENT, RNN_MEMORY),
        ("LSTM (forget bias 1)", LSTM_GRADIENT, LSTM_MEMORY),
        ("GRU (update bias -1)", gru_norms[0], memory(gru_norms)),
        ("GRU (update bias 0)", gru_flow(100, 0.0)[0][0], mem[0.0])]
for name, grad, steps in rows:
    print(f"  {name:<26}{grad:>12.2e}{steps:>6} steps")
assert rows[2][2] > LSTM_MEMORY             # 19 against 15
assert rows[3][2] < RNN_MEMORY              # 8 against 9 - BELOW a plain RNN
print()
print("Read the last row. Two gates and 3x the parameters of an RNN, and with")
print("the bias left at 0 it remembers LESS than the RNN. Gating is permission")
print("to remember, not remembering - the same finding as lesson 062.")

## Part D — Parameters, and wall clock

In [ ]:
import time

for f_dim, h_dim in ((100, 150), (2, 32), (100, 128)):
    base = (f_dim + h_dim) * h_dim + h_dim
    print(f"  F={f_dim:<4} H={h_dim:<4} RNN {base:>8,}  GRU {3 * base:>9,}  "
          f"LSTM {4 * base:>9,}")
    assert 3 * base == int(4 * base * 0.75)          # exactly 25% fewer
print("  GRU is exactly 3 blocks against the LSTM's 4 - 25% fewer, at any size")

def benchmark(kind, steps=200, reps=30):
    r = np.random.default_rng(0)
    n = 3 if kind == "gru" else 4
    Wb = [r.normal(0, 0.3, (F + H, H)) for _ in range(n)]
    X = r.normal(0, 1, (steps, F))
    start = time.perf_counter()
    for _ in range(reps):
        hh, CC = np.zeros(H), np.zeros(H)
        for t in range(steps):
            zz = np.concatenate([hh, X[t]])
            g = [sigmoid(zz @ w) for w in Wb]
            if kind == "gru":
                hh = (1 - g[1]) * hh + g[1] * np.tanh(zz @ Wb[2])
            else:
                CC = g[0] * CC + g[1] * np.tanh(zz @ Wb[2])
                hh = g[3] * np.tanh(CC)
    return (time.perf_counter() - start) / reps * 1000

gru_ms, lstm_ms = benchmark("gru"), benchmark("lstm")
print(f"\n  200 steps: GRU {gru_ms:.1f} ms   LSTM {lstm_ms:.1f} ms   "
      f"GRU {(1 - gru_ms / lstm_ms) * 100:.0f}% faster")
assert gru_ms < lstm_ms
print("  25% fewer parameters buys well under 25% less time: parameter count")
print("  and wall clock are related, not proportional.")

## What this notebook established

- The GRU merges keeping and writing into one gate — $(1-z)$ and $z$ sum to 1 —
  and has no cell state.
- Gradient 99 steps back and memory: **1.5e-29 / 8 steps** at update bias 0,
  **6.6e-13 / 19 steps** at bias −1.
- Side by side: RNN 9 steps, LSTM 15, GRU 19 — but the **ungated GRU at 8 steps
  is below the plain RNN's 9**, with three times the parameters.
- The GRU is **exactly 25%** smaller than an LSTM at every size, and measured
  here about 13–16% faster per step.

## Exercises

1. Set the reset gate to 1 permanently. What does the GRU become?
2. Find the update bias at which the GRU matches the LSTM's 15 steps. Is it
   cheaper to get there this way?
3. Benchmark at H = 128 instead of 20. Does the speed gap widen or narrow?